**Criação da camada Bronze:**

Extraindo os dados e colocando em path temporário (landing zone)

In [180]:
import requests
import zipfile
import os
from pyspark.sql import SparkSession

URL = "https://dadosabertos.ans.gov.br/FTP/PDA/informacoes_consolidadas_de_beneficiarios-024/202508/pda-024-icb-TO-2025_08.zip"
EXTRACT_DIR = "dados/landing_zone/"
FILE = "dados/landing_zone/pda-024-icb-TO-2025_08.zip"

# Cria a pasta onde será extraído
os.makedirs(EXTRACT_DIR, exist_ok=True)

try:
    #Faz um get na URL de origem com stream ativado (para caso o arquivo ser grande)
    response = requests.get(URL, stream=True)
    with open(FILE, 'wb') as file:
        file.write(response.content)
    print("Fonte de dados obtida com sucesso")

    #Se der sucesso, descompacta o zip para pegar o csv
    with zipfile.ZipFile(FILE, 'r') as zip_ref:
        zip_ref.extract("pda-024-icb-TO-2025_08.csv", EXTRACT_DIR)    
    CSV_PATH = os.path.join(EXTRACT_DIR, "pda-024-icb-TO-2025_08.csv")
    print("Arquivo CSV extraído com sucesso")

except Exception as e:
    print(f"Erro na obtencao dos dados. {e}")
    exit()

Fonte de dados obtida com sucesso
Arquivo CSV extraído com sucesso


Configurando Spark:

In [181]:
spark = SparkSession.builder \
    .appName("SparkBeneficiarios") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Persistindo o CSV na camada bronze:

In [171]:
BRONZE_DIR = "dados/bronze"

df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ";") \
    .load(CSV_PATH)

df_bronze.write \
    .mode("overwrite") \
    .csv(BRONZE_DIR, header=True, sep=";")
    #.parquet(BRONZE_DIR) -- Para otimizar, poderia ter já convertido em parquet na bronze

print("Camada bronze criada com sucesso!")

Camada bronze criada com sucesso!


**Criação da camada Silver:**

Leitura do arquivo CSV da camada bronze, tipagem e mascaramento:

In [172]:
spark.read.csv(BRONZE_DIR, header=True, sep=";") \
    .createOrReplaceTempView("bronze_view")

silver_sql = f"""
CREATE OR REPLACE TEMPORARY VIEW silver_view AS
SELECT 
    CAST(CD_OPERADORA AS STRING) AS CODIGO_OPERADORA,
    CAST(NM_RAZAO_SOCIAL AS STRING) AS NOME_RAZAO_SOCIAL,
    CAST(NM_MUNICIPIO AS STRING) AS NOME_MUNICIPIO,
    CAST(DE_FAIXA_ETARIA AS STRING) AS FAIXA_ETARIA,
    CAST(QT_BENEFICIARIO_ATIVO AS INT) AS BENEFICIARIOS_ATIVOS,
    CONCAT(REPEAT('X', 5), SUBSTR(CAST(CD_PLANO AS STRING), -4)) AS CODIGO_PLANO
FROM
    bronze_view
"""

spark.sql(silver_sql)
spark.sql("SELECT * FROM silver_view").show(5)

+----------------+--------------------+--------------------+------------+--------------------+------------+
|CODIGO_OPERADORA|   NOME_RAZAO_SOCIAL|      NOME_MUNICIPIO|FAIXA_ETARIA|BENEFICIARIOS_ATIVOS|CODIGO_PLANO|
+----------------+--------------------+--------------------+------------+--------------------+------------+
|          313084|COOPERATIVA DE TR...|         Nova Olinda|  1 a 4 anos|                   1|   XXXXX1165|
|          313084|COOPERATIVA DE TR...| Formoso do Araguaia|18 a 19 anos|                   1|   XXXXX0191|
|          313084|COOPERATIVA DE TR...|Santa Terezinha d...|  5 a 9 anos|                   1|   XXXXX4176|
|          313084|COOPERATIVA DE TR...|Bandeirantes do T...|35 a 39 anos|                   2|   XXXXX9998|
|          005711| BRADESCO SAÚDE S.A.|               Almas|18 a 19 anos|                   1|   XXXXX6138|
+----------------+--------------------+--------------------+------------+--------------------+------------+
only showing top 5 rows



Limpeza de nulos, vazios e 0:

In [173]:
COLUNAS = [
    "CODIGO_OPERADORA", "NOME_RAZAO_SOCIAL", "NOME_MUNICIPIO", "FAIXA_ETARIA"
]

#Loop para adicionar todos os campos na condição de nulos e vazios
condicao_limpeza_lista = []
for coluna in COLUNAS:
    condicao = f"{coluna} IS NOT NULL AND {coluna} != ''"
    condicao_limpeza_lista.append(condicao)
condicao_limpeza = " AND ".join(condicao_limpeza_lista)

#No select faz tipagem dos dados e mascaramento do cd_plano no formato XXXXX1234
silver_sql_filter = f"""
SELECT *
FROM
    silver_view
WHERE
    BENEFICIARIOS_ATIVOS > 0
    AND {condicao_limpeza}
"""

df_silver = spark.sql(silver_sql_filter)

Persistindo o CSV na camada silver:

In [174]:
SILVER_DIR = "dados/silver"

df_silver.write \
    .mode("overwrite") \
    .parquet(SILVER_DIR)

print("Camada silver criada com sucesso!")

Camada silver criada com sucesso!


**Criação da camada Gold**

Otimização e gravação particionada para consultas futuras:

In [175]:
GOLD_DIR = "dados/gold"

#Feito pre-calculo do total de beneficiarios agrupando por operadora, municipio e faixa etaria
gold_sql = f"""
SELECT
    CODIGO_OPERADORA,
    NOME_RAZAO_SOCIAL,
    NOME_MUNICIPIO,
    FAIXA_ETARIA,
    SUM(BENEFICIARIOS_ATIVOS) AS TotalBeneficiarios
FROM
    parquet.`{SILVER_DIR}`
GROUP BY CODIGO_OPERADORA, NOME_RAZAO_SOCIAL, NOME_MUNICIPIO, FAIXA_ETARIA
"""

df_gold = spark.sql(gold_sql)

#Gravacao com particionamento por operadora
df_gold.write \
    .mode("overwrite") \
    .partitionBy("CODIGO_OPERADORA") \
    .parquet(GOLD_DIR)


Criação de caching em memória para responder às perguntas:

In [176]:
df_gold = spark.read.parquet(GOLD_DIR)
df_gold.cache() 
df_gold.createOrReplaceTempView("gold_view")

**Perguntas e Respostas**

a) Quais são as 5 operadoras com maior número de beneficiários ativos?

In [177]:
consulta_a = """
SELECT
    NOME_RAZAO_SOCIAL,
    SUM(TotalBeneficiarios) as BeneficiariosAtivos
FROM
    gold_view
GROUP BY CODIGO_OPERADORA, NOME_RAZAO_SOCIAL
ORDER BY BeneficiariosAtivos desc
LIMIT 5
"""
spark.sql(consulta_a).show(truncate=False)

+--------------------------------------------------------------+-------------------+
|NOME_RAZAO_SOCIAL                                             |BeneficiariosAtivos|
+--------------------------------------------------------------+-------------------+
|PREVIDENT ASSISTÊNCIA ODONTOLÓGICA S.A                        |67430              |
|ODONTOPREV S/A                                                |52846              |
|UNIMED PALMAS COOPERATIVA DE TRABALHO MÉDICO                  |29637              |
|BRADESCO SAÚDE S.A.                                           |16837              |
|COOPERATIVA DE TRABALHO MEDICO DE ARAGUAÍNA - UNIMED ARAGUAÍNA|14366              |
+--------------------------------------------------------------+-------------------+



b) Qual é a faixa etária com mais beneficiários e quantos são?

In [178]:
consulta_b = """
SELECT
    FAIXA_ETARIA,
    SUM(TotalBeneficiarios) as BeneficiariosAtivos
FROM
    gold_view
GROUP BY FAIXA_ETARIA
ORDER BY BeneficiariosAtivos desc
LIMIT 1
"""
spark.sql(consulta_b).show(truncate=False)

+------------+-------------------+
|FAIXA_ETARIA|BeneficiariosAtivos|
+------------+-------------------+
|35 a 39 anos|27677              |
+------------+-------------------+



c) Liste, de forma decrescente, a quantidade de beneficiários por município

In [179]:
consulta_c = """
SELECT
    NOME_MUNICIPIO,
    SUM(TotalBeneficiarios) AS BeneficiariosAtivos
FROM
    gold_view
GROUP BY NOME_MUNICIPIO
ORDER BY BeneficiariosAtivos DESC
"""
spark.sql(consulta_c).show(truncate=False)

+---------------------+-------------------+
|NOME_MUNICIPIO       |BeneficiariosAtivos|
+---------------------+-------------------+
|Palmas               |128401             |
|Araguaína            |38027              |
|Gurupi               |18325              |
|Porto Nacional       |16113              |
|Paraíso do Tocantins |7770               |
|Colinas do Tocantins |4814               |
|Pedro Afonso         |4711               |
|Guaraí               |3964               |
|Dianópolis           |2868               |
|Miracema do Tocantins|2733               |
|Araguatins           |2484               |
|Xambioá              |2366               |
|Tocantinópolis       |2220               |
|Taguatinga           |2171               |
|Formoso do Araguaia  |2090               |
|Alvorada             |1746               |
|Almas                |1282               |
|Natividade           |1268               |
|Arraias              |1166               |
|Miranorte            |1154     

In [ ]:
spark.stop()